In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils import (
    BereaPatchDataset,
    DigitalCorePipeline,
    FiLMRoutedUNet3D,
    check_required_dependencies,
)

check_required_dependencies()
device = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = ROOT / "outputs" / "networks"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


ROOT: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct device: cuda


In [2]:
CUBE_SIZE = 128
SPLIT = "train"
MAX_SAMPLES = 20
USE_TARGET_MASK = True
THRESHOLD = 0.5
VOXEL_SIZE_M = 2.25e-6
DOMAIN_SIZE = (CUBE_SIZE * VOXEL_SIZE_M, CUBE_SIZE * VOXEL_SIZE_M, CUBE_SIZE * VOXEL_SIZE_M)
SEG_CHECKPOINT = ROOT / "models" / "film_routed_unet3d_best.pth"


In [3]:
dataset = BereaPatchDataset(ROOT, split=SPLIT, cube_size=CUBE_SIZE, use_raw_gray=False, noise_types=["none"])
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

seg_model = None
if not USE_TARGET_MASK:
    seg_model = FiLMRoutedUNet3D(base_channels=16, ctx_dim=64).to(device)
    checkpoint = torch.load(SEG_CHECKPOINT, map_location=device)
    seg_model.load_state_dict(checkpoint["model_state_dict"])

pipeline = DigitalCorePipeline(
    segmentation_model=seg_model,
    device=device,
    threshold=THRESHOLD,
    voxel_size=VOXEL_SIZE_M,
    mu=1.0e-3,
)
print("samples:", len(dataset))


samples: 265


In [4]:
rows = []

for sample_id, batch in enumerate(tqdm(loader, desc="extract networks")):
    if MAX_SAMPLES is not None and sample_id >= MAX_SAMPLES:
        break

    if USE_TARGET_MASK:
        cube = batch["y"][0, 0].numpy() > 0.5
        input_is_pore_mask = True
    else:
        cube = batch["x"][0].numpy()
        input_is_pore_mask = False

    try:
        result = pipeline.run_cube(
            cube,
            input_is_pore_mask=input_is_pore_mask,
            domain_size=DOMAIN_SIZE,
            include_ph=True,
            compute_openpnm_baseline=True,
        )
    except Exception as exc:
        print(f"sample {sample_id} skipped: {exc}")
        continue

    coord = batch["coord"][0].tolist()
    result.network.metadata.update({
        "sample_id": sample_id,
        "coord": coord,
        "openpnm_k": result.permeability.k_openpnm,
    })
    out_path = OUT_DIR / f"network_{SPLIT}_{sample_id:04d}.pt"
    torch.save(result.network, out_path)

    row = {
        "sample_id": sample_id,
        "path": str(out_path),
        "z": coord[0],
        "y": coord[1],
        "x": coord[2],
        **result.network.metadata,
    }
    if result.permeability.k_openpnm is not None:
        row.update({f"openpnm_{k}": v for k, v in result.permeability.k_openpnm.items()})
    rows.append(row)

summary = pd.DataFrame(rows)
summary_path = OUT_DIR / f"network_{SPLIT}_summary.csv"
summary.to_csv(summary_path, index=False)
summary


extract networks:   8%|▊         | 20/265 [03:52<47:32, 11.64s/it] 


,sample_id,path,z,y,x,num_pores,num_throats,node_feature_dim,edge_feature_dim,ph_summary,coord,openpnm_k,openpnm_kx,openpnm_ky,openpnm_kz
0,0,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,0,0,0,378,566,12,12,"[377.0, 0.008982167579233646, 6.34981843177229...","[0, 0, 0]","{'kx': 2.198275785811183e-13, 'ky': 1.43602103...",2.198276e-13,1.436021e-13,1.422828e-13
1,1,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,0,0,128,360,489,12,12,"[359.0, 0.008508502505719662, 5.17018852406181...","[0, 0, 128]","{'kx': 1.1084026873664714e-13, 'ky': 4.7893597...",1.108403e-13,4.789360e-14,2.108055e-14
2,2,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,0,0,256,217,306,12,12,"[216.0, 0.005286002531647682, 6.35760807199403...","[0, 0, 256]","{'kx': 3.001297556997469e-14, 'ky': 2.28918799...",3.001298e-14,2.289188e-14,1.115487e-13
3,3,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,0,0,384,347,504,12,12,"[346.0, 0.008195172064006329, 5.48966563655994...","[0, 0, 384]","{'kx': 1.7271990124265905e-13, 'ky': 1.4712882...",1.727199e-13,1.471288e-14,1.136053e-13
4,4,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,0,0,640,347,520,12,12,"[346.0, 0.008820229209959507, 6.26994515187107...","[0, 0, 640]","{'kx': 1.0247651327267543e-13, 'ky': 8.1200028...",1.024765e-13,8.120003e-14,1.098351e-13
5,5,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,0,0,768,383,564,12,12,"[382.0, 0.009069332852959633, 5.71291529922746...","[0, 0, 768]","{'kx': 1.0942233986675567e-13, 'ky': 3.8653507...",1.094223e-13,3.865351e-14,1.249227e-13
6,6,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,0,128,0,415,635,12,12,"[414.0, 0.009551878087222576, 5.74823534407187...","[0, 128, 0]","{'kx': 1.4925310723323522e-13, 'ky': 9.3333875...",1.492531e-13,9.333388e-14,2.357661e-13
7,7,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,0,128,256,394,600,12,12,"[393.0, 0.009471358731389046, 5.23858652741182...","[0, 128, 256]","{'kx': 1.284898611992001e-13, 'ky': 1.30445558...",1.284899e-13,1.304456e-13,2.090094e-13
8,8,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,0,128,384,364,522,12,12,"[363.0, 0.008635059930384159, 5.48005700693465...","[0, 128, 384]","{'kx': 1.018735580685695e-13, 'ky': 9.13578957...",1.018736e-13,9.135790e-15,6.480306e-14
9,9,c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\outputs\ne...,0,128,512,334,445,12,12,"[333.0, 0.0074641164392232895, 7.5281372119206...","[0, 128, 512]","{'kx': 2.614964449966663e-14, 'ky': 4.61998637...",2.614964e-14,4.619986e-14,6.262868e-14
